# **Cheminformatics with RDKit**

# **Goals of Today**


*   **Chemistry**: Chemical descriptors
*   **Computation**: Working with chemical data with RDKit





#**0.Installations**

Before we proceed, let us first install the following packages:

*   `RDkit`
*   `Chemspipy`


In [ ]:
# install conda
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# Install rdkit
!mamba install rdkit

In [ ]:
# Install chemspipy
!pip install chemspipy

#**1. Introduction**

*Cheminformatics* can be thought of as the intersection of data science, computer science, and chemistry as a means of better understanding and solving chemical problems. This chapter introduces a popular and versatile Python cheminformatics library known as RDKit which is useful for tasks such as:

- Visualizing molecules
- Reading SMILES or InChI molecular representations
- Quantifying structural features in molecules such as number of rings or hydrogen bond donors
- Generating all possible stereoisomers of a molecular structure
- Filtering molecules based on structural features

This is a popular library for those in chemical computing research with examples of its use being relatively easy to find in the chemical literature.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem.Draw import SimilarityMaps
from rdkit.Chem import rdFingerprintGenerator

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RDKit is composed of a number of modules including, but not limited to, the following.

**Key Modules and Submodules in the RDKit Library**

| Module/Submodule      | Description                |
|:-----------:|:---------------------------|
|`Chem`       |General purpose tools for chemistry. The [RDKit website](https://www.rdkit.org/docs/source/rdkit.Chem.html#module-rdkit.Chem) describes it as "A module for molecules and stuff". |
|`Chem.AllChem` | Submodule containing more specialized or less often used features; needs to be imported separatly from `Chem` |
|`Chem.Descriptors` | Submodule for quantifying molecular features |
|`Chem.Draw` | Submodule for visualizing molecules |
|`ML`         |Machine learning tools      |

The `Chem` and `ML` modules are the major modules in RDKit, but for this chapter, we will only be focusing on the `Chem` module which has already been imported above.


#**2. Loading Molecular Representations into RDKit**

We have discussed how we can identify molecules using SMILES, InChiI, MOL files, and XYZ files. RDKit is able to load a molecule from all of these repesentations (and more).

Once the molecule is loaded, you can also generate the other representations of it.

The functions below can read and write molecular structures from a variety of formats including SMILES, InChI, and MOL files. When reading these molecular structures, a Molecule object (RDKit-specific class of object) is generated.

**Functions for Loading Molecular Structures**

| Function      | Description                |
|:-------------:|---------------------------|
|`Chem.MolFromSmiles()`   | Generates a Molecule object from SMILES representation |
|`Chem.MolToSmiles()` | Generates SMILES representation from a Molecule object |
|`Chem.inchi.MolFromInchi()` | Generates a Molecule object from InChI representation |
|`Chem.inchi.MolToInchi()` | Generates InChI representation from a Molecule object |
|`MolFromMolFile()` | Generates a Molecule object from a MOL file |



##**Demonstration: Aspirin**

As an example, we will load the structure of aspirin (acetylsalicylic acid) using the `Chem.MolFromSmiles()` function from the `Chem` module.

In [ ]:
aspirin = Chem.MolFromSmiles('O=C(C)Oc1ccccc1C(=O)O')
aspirin

If we check the object type, we find that it is a Molecule (`rdchem.Mol`) RDKit object.

In [ ]:
type(aspirin)

RDKit can generate other molecular representations such as InChI from the Molecule object as demonstrated below.

In [ ]:
Chem.inchi.MolToInchi(aspirin)

# **3 Visualizing Chemical Structures**

In the above examples, RDKit provided an image of the molecule simply by Jupyter running the Molecule object. By default, this generates a rather small and low resolution image. To generate a sharper image, like above, one may change the settings to produce SVG (Scalable Vector Graphic) images which are a vector graphic format.



##**3.1 Single Chemical Structures**

However, simply running the Molecule object does not provide easy control over the image. Here, we explore some other ways you may view the molecule.

To view the molecule, we can use the `Chem.Draw.MolToImage(Mol)` function which takes one required positional arguments of the Molecule object (`Mol`). Optional keyword arguments can be used to set other parameters such as the image size (`size=`) in pixels.

In [ ]:
Chem.Draw.MolToImage(aspirin, size=(200,200))

If we want to save the image to a file, this is accomplished using the `Chem.Draw.MolToFile()` function which requires two pieces of information - the Molecule object and the name of the new file as a string.

~~~python
Chem.Draw.MolToFile(mol_object, 'file_name.png', size=(width, height), imageType='png')
~~~

Other optional parameters include the `size=` which is a tuple that takes the width and height, respectively, in pixels, and the `imageType=` accepts a string to designate the file format ('png' or 'svg').
The PNG file formate is a great general-purpose raster file format. Unless you know you need a different file format, this is often a good choice. The SVG file format is a vector format which makes it easily editable in software applications such as Adobe Illustrator.


It is important that the extension (e.g., ".png") matches the `imageType=` argument or else your computer may have difficulties opening the file.


In [ ]:
Chem.Draw.MolToFile(aspirin, 'aspirin.svg',
                    size=(500,500),
                    imageType='svg')

In [ ]:
!ls

In [ ]:
# you may download the file to your computer
from google.colab import files
files.download('aspirin.svg')

## **3.2 Grids of Chemical Structures**

Whenever we are dealing with collections of molecules, it may be helpful to generate an image that includes multiple molecular structures known as a grid. In this exercise, we will load the SMILES strings of the twenty common amino acids from a text file using pandas and then load the Molecule objects for each structure into a single list called `AminoAcids`.

###**In-class exercise: Amino Acids Database**

Below is the list of twenty amino acids. Build a `Pandas.DataFrame` to include a new column with the SMILES information. Note that you can obtain SMILES from a chemical database such as ChemSpider

In [ ]:
amino_acids = [
    "alanine", "arginine", "asparagine", "aspartic acid", "cysteine",
    "glutamine", "glutamic acid", "glycine", "histidine", "isoleucine",
    "leucine", "lysine", "methionine", "phenylalanine", "proline",
    "serine", "threonine", "tryptophan", "tyrosine", "valine"
]

df = pd.DataFrame(amino_acids, columns=['name'])
df

In [ ]:
from chemspipy import ChemSpider

In [ ]:
cs = ChemSpider('<INSERT YOUR 32-CHARACTER KEY HERE>')

In [ ]:
# Code Here to insert the SMILES information into df


In [ ]:
# Create a list of Amino Acids molecule objects
AminoAcids = [Chem.MolFromSmiles(SMILES) for SMILES in df['SMILES']]
AminoAcids

To generate the grid, we will use the `MolsToGridImage()` function from the `Chem.Draw` submodule. This function requires one positional argument of an array-like object (e.g., list, tuple, ndarray, etc.) containing the Molecule objects. Other optional keyword arguments include the number of molecules per row (`molsPerRow=`), the pixel dimensions of each molecule (`subImgSize=`), labels below each molecule (`legends=`), and the ability to make images SVG format (`usesSVG=`). The image dimensions only matter if using a raster image format and requires a tuple with the width and height in that order. The `legends=` argument requires an array-like object with the labels in the same order as the object containing the Molecule objects.

In [ ]:
Chem.Draw.MolsToGridImage(AminoAcids,
                          molsPerRow=4,
                          subImgSize=(200,200),
                          legends=list(df['name']),
                          useSVG = False,
                          )

# **4. Chemical Descriptors**

RDKit can be use to determine a number of key physical properties of molecules known as *descriptors* using the `Chem.Descriptor` module. These can be useful for generating features for a large number of molecules for machine learning or understanding structural trends in a body of chemical compounds.

## **4.1 Molecular Features**

There are numerous descriptor functions available which are callable using `Chem.Descriptors.method()` where `method()` is the name of a descriptor function that accepts an RDKit Molecule object and returns a numerical value. Below are a few examples of descriptor functions, with a more complete list is available on the [RDKit website](https://www.rdkit.org/docs/GettingStartedInPython.html#list-of-available-descriptors).

**Examples of Molecular Descriptors**

| Function    |  Description |
|:-------------:|---------------------------|
|`MolWt` | Molecular weight, assumes natural isotopic distribution  |
|`HeavyAtomCount()` | Number of non-hydrogen atoms |
|`NOCount()` | Number of N and O atoms |
|`NumAliphaticRings()` | Number of aliphatic rings
|`NumAromaticRings()` | Number of aromatic rings |
|`NumSaturatedRings()` | Number of saturated rings |
|`NumHAcceptors()`   | Number of hydrogen bond acceptors   |
|`NumHDonors()` | Number of hydrogen bond donors |
|`NumRadicalElectrons()`   |Number of radical electrons |
|`NumValenceElectrons()` | Number of valence electrons |
|`NumRotatableBonds()` | Number of rotatable bonds |
|`RingCount()` | Number of rings |



###**Demonstration: molecular descriptor of aspirin**

We have earlier loaded aspirin. Let us take a look at its molecular descriptors.

In [ ]:
aspirin

In [ ]:
# molecular weight
Chem.Descriptors.MolWt(aspirin)

In [ ]:
# number of rings
Chem.Descriptors.RingCount(aspirin)

In [ ]:
# number of aromatic rings
Chem.Descriptors.NumAromaticRings(aspirin)

In [ ]:
# number of valence electrons
Chem.Descriptors.NumValenceElectrons(aspirin)

###**In-class exercise: molecular descriptors of amino acids**

Build onto the `pandas.DataFrame` for the amino acids, include new columns by including data on the molecular weight, number of aromatic rings, and numbers of hydrogen donors and acceptors.

In [ ]:
# [Code Here]

##**4.2 Quantifying Functional Groups**

Among the descriptor methods is a long list of functions that look like `fr_group()` where `group` is the name or abbreviation of a chemical functional group. These functions return an integer quantification of that functional group present in the molecule. A table with a few examples is provided below, but there are over 80 of these functions available in RDKit.

**Examples of Methods to Quantify Functional Groups**

| Function    |  Functional Group |
|:-----------:|---------------------------|
|`fr_Al_OH()` | Aliphatic alcohols  |
|`fr_aldehyde()` | Aldehydes |
|`fr_amide()` | Amide |
|`fr_C_C()`   | Carbonyl oxygens   |
|`fr_guanido()` | Guanidine |
|`fr_NH0()`   | Amines with 0 H's (i.e., tertiary) |
|`fr_phenol()` | Phenol |
|`fr_phos_ester()` | Phosphoric ester |
|`fr_SH()` | Thiol |





###**Demonstration: functional groups of aspirin**

In [ ]:
# number of benzene rings
Chem.Descriptors.fr_benzene(aspirin)

In [ ]:
# number of alphatic alcohols
Chem.Descriptors.fr_Al_OH(aspirin)

In [ ]:
# number of aromatic carboxyls
Chem.Descriptors.fr_Ar_COO(aspirin)

In [ ]:
# number of esters
Chem.Descriptors.fr_ester(aspirin)

###**In-class exercise: functional groups of amino acids**

Build onto the `pandas.DataFrame` for the amino acids, include new columns by including data on the numbers of aliphatic alcohols, aldehydes, and amines.

In [ ]:
# [Code here]

# **5 Searching Molecules for Structural Patterns**

Molecules can be searched for key structural features using the `HasSubstructMatch()` method which returns `True` or `False` depending if a structural pattern exists in a molecule or not. This function requires two RDKit Molecule objects - one Molecule object (`molecule`) is checked for the presence of the other Molecule object structure (`substructure`) as shown below.


**SMILES Bond Order Notation**

| SMILES Bond | Bond Type |
|:----------:|------------|
| - (or nothing) | Single |
| = | Double |
| # | Triple |
|: | Aromatic |

## **Demonstration: functional groups in acetone**

As an example, we will look for the presence of a carbonyl (i.e., C=O bond) in acetone, so the substructure that we will search for is a `C=O`.

In [ ]:
acetone = Chem.MolFromSmiles('CC(=O)C')
acetone

In [ ]:
substructure = Chem.MolFromSmiles('C=O')
acetone.HasSubstructMatch(substructure)

In [ ]:
substructure = Chem.MolFromSmiles('CO')
acetone.HasSubstructMatch(substructure)

## **Demonstration: sub-structure on amino acid**

For a more interesting set of examples, we can search our collection of 20 common amino acids (see [section 15.2.2](15.2.2)) for key substructures. We will start by using glycine, the simplest of the common amino acids, as the substructure which should return all 20 amino acids. As an extra step below, we will also orient all the amino acids in the same way with respect to the substructure. That is, the substructural element that we are searching for in each amino acid will be oriented the same way for all 20 amino acids.

In [ ]:
# define substruture
substructure= Chem.MolFromSmiles('NC(C(=O)[OH])')
substructure

In [ ]:
# search for matching substructures
matching_amino_acids = [AA for AA in AminoAcids if AA.HasSubstructMatch(substructure)]
matching_amino_acids

In [ ]:
# orients common substructures the same way
AllChem.Compute2DCoords(substructure)
for amino_acid in matching_amino_acids:
    _ = AllChem.GenerateDepictionMatching2DStructure(amino_acid, substructure)

# generates grid of matching molecules
Chem.Draw.MolsToGridImage(matching_amino_acids,
                          molsPerRow=4,
                          subImgSize=(200,200),
                          legends=list(df['name']),
                          useSVG=False)

Notice that the search returns all 20 amino acids, because they all have the substructure (that's what defines an amino acid). Notice how the core structure of all amino acids are oriented the same direction.

##**In-class exercise: benzene ring on amino acid**

Now let us try something a little more interesting by search for all amino acids with a benzene ring in them. The substructural bonding pattern in this case is benzene itself, and the three aromatic amino acids are returned.

In [ ]:
# Search for benzene substructure in amino acids
# [Code here]

# **6. Atoms and Bond (Optional)**



## **6.1 Atom Methods**
RDKit allows access to information on specific atoms and bonds through the `GetAtoms()` and `GetBonds()` methods, respectively. These functions return a sequence type of object that can be iterated through using a `for` loop to access individual atoms or bonds. Using the following methods, the user can
access or even modify various pieces of information about the atoms or bonds. The table below contain some key functions for working with atoms and bonds.

**Select Atom Methods**

| Function | Description |
|:----------:|---------------|
|`GetDegree()` | Returns number of atoms bonded directly to it, includes hydrogens only if they are explicitly defined |
|`GetAtomicNum()` | Returns atomic number |
|`GetChiralTag()` | Determines if the atom is a chiral center and CW or CCW designation |
|`GetFormalCharge()` | Returns formal charage of atom   |
|`GetHybridization()` | Returns hybridization of atom     |
|`GetIsAromatic()`   | Returns bool as to whether atom is aromatic    |
|`GetIsotope()`   |  Returns isotope number if designated, otherwise returns `0`   |
|`GetNeighbors()` | Returns tuple of directly bonded atoms  |
|`GetSymbol()` |  Returns atomic symbols as a string |
|`GetTotalNumHs()` | Returns number of hydrogens bonded to the atom |
|`IsInRing()` |  Returns bool designating if the atom is in a ring  |
|`SetAtomicNum()`| Sets the atomic number to user defined value |
|`SetFormalCharge()` |  Sets formal charge to user defined value |
|`SetIsotope()` |  Sets isotope to user defined integer value  |



###**Demonstration: atom method on aspirin**

As an example, let's look at the atoms in aspirin.

In [ ]:
aspirin = Chem.MolFromSmiles('O=C(C)Oc1ccccc1C(=O)O')
aspirin

If we generate a list populated with the degrees of atoms (i.e., number of other atoms bonded directly to it), you may notice that there are no 4 values even though the methyl (i.e., -CH$_3$) carbon should have four atoms attached to it. This is because the hydrogens are not explicitly designated in the structure (i.e., the are implicit), so they are not counted.

In [ ]:
[atom.GetDegree() for atom in aspirin.GetAtoms()]

We can count the number of implicit hydrogens using the `GetNumImplicitHs()` method, and the third value is a `3` making it the methyl carbon.

In [ ]:
[atom.GetNumImplicitHs() for atom in aspirin.GetAtoms()]

We can also use these atom methods to change values and attributes of various atoms. For example, we can set the isotopes of the carbonyl carbons (i.e., C=O) to $^{13}$C. This is accomplished with the following code that iterates through all the atoms and finds the carbonyl carbons by testing for atoms that have an atomic number of 6, are not aromatic, and have no hydrogens and then setting the isotope value to 13. The molecular weight is calculated before and after the isotopes are changed for comparison.

In [ ]:
print(Chem.Descriptors.MolWt(aspirin))
aspirin

In [ ]:
for atom in aspirin.GetAtoms():
    if atom.GetAtomicNum() == 6 and \
        not atom.GetIsAromatic() and \
        atom.GetTotalNumHs() == 0:

        atom.SetIsotope(13)

In [ ]:
print(Chem.Descriptors.MolWt(aspirin))
aspirin

The molar mass has increased due to two of the carbon atoms being isotopically labeled, and we can see in the image which of the two carbons were isotopically labeled. It is worth nothing that the molecular weight before isotopically labeling assumes a natural distribution of isotopes which for carbon is 98.9% $^{12}$C and 1.1% $^{13}$C. In the isotopically labeled structure, the two carbonyl carbons are 100% $^{13}$C.



## **6.2 Bond Methods**
Using bond methods, we can perform analogous types of operations except that bonds have different attributes than atoms. A table of selected bond methods is provided below.

** Select Bond Methods**

| Function | Description |
|:----------:|---------------|
|`GetBeginAtom()` | Returns first atom in bond |
|`GetEndAtom()` | Returns second atom in bond |
|`GetBondType()` | Returns type of bond (e.g., SINGLE, DOUBLE, AROMATIC) |
|`GetIsAromatic()` | Returns bool as to whether bond is aromatic |
|`GetIsConjugated()` | Returns bool as to wether bond is conjugated |
|`IsInRing()` | Returns bool as to wether bond is in ring  |
|`SetBondType()` | Sets bond type |
|`SetIsAromatic()` | Sets bool designating if a bond is aromatic |



###**Demonstration: bond methods on acetone**

As a demonstration, we will examine the bonds in the structure of acetone and change the carbonyl double bond to a single bond. This is done by searching for a double bond, setting it to a single bond, and then changing the formal charges of the atoms attached to that bond.

In [ ]:
acetone

In [ ]:
for bond in acetone.GetBonds():
    if bond.GetBondType() == Chem.BondType.DOUBLE:
        bond.SetBondType(Chem.BondType.SINGLE)
        end = bond.GetEndAtom().SetFormalCharge(-1)
        begin = bond.GetBeginAtom().SetFormalCharge(+1)
acetone


# **Further Reading**

This set of note is largely based on the tutorial available in [Scientific Computing for Chemists with Python](https://weisscharlesj.github.io/SciCompforChemists/notebooks/introduction/intro.html). Other useful resources can be found online:

1. RDKit: Open-Source Cheminformatics Software. [https://www.rdkit.org/](https://www.rdkit.org/) (free resource)
2. The RDKit Book (collection of examples). [https://www.rdkit.org/docs/RDKit_Book.html](https://www.rdkit.org/docs/RDKit_Book.html) (free resource)

# **Debrief: What have we learned today?**

Let us reflect on what are the new skills we have learned today. We will classify the skills into chemical knowledge and computation skills.

**New chemical knowledge today:**

*   List response
*   List response

**New computational skill today:**

*   List response


## **How can I make the class better?**

Please let me know [here](https://docs.google.com/forms/d/e/1FAIpQLSd6iRTXWhMHmc0MqXXiAWdglDqEa0mKcXQRXm6LyD4HS-6v7g/viewform)!